In [6]:
# Imports i wczytanie danych

import pandas as pd
import numpy as np
from datetime import datetime
import sys
from pathlib import Path

sys.path.append('.')
from run_etl import load_raw_excel, clean_raw

RAW_PATH = "../data/raw/online_retail_II.xlsx"
raw = load_raw_excel(RAW_PATH)
print(f"RAW: {len(raw):,} wierszy")

[2025-09-01 14:58:02] Ładowanie surowych danych z: ../data/raw/online_retail_II.xlsx
[2025-09-01 14:59:12] Wczytano 1,067,371 rekordów z 2 arkuszy
RAW: 1,067,371 wierszy


In [8]:
# Definicja clean_raw_v2

def clean_raw_v2(df_raw: pd.DataFrame) -> pd.DataFrame:
    req = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
    missing = [c for c in req if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Brak kolumn: {missing}")

    df = df_raw.copy()

    # Typy danych
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
    df["Price"] = pd.to_numeric(df["Quantity"], errors="coerce")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

    # Normalizacja stringów
    for c in ["Invoice", "StockCode", "Description", "Country"]:
        df[c] = df[c].astype(str).str.strip()

    df["Customer ID"] = (
        df["Customer ID"].astype (str).str.strip()
        .replace({"nan": np.nan, "": np.nan})
    )

    # Deduplikacja
    before = len(df)
    df = df.drop_duplicates(subset=["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"], keep="first")

    # KLUCZOWA ZMIANA: Filtruj tylko oczywiste błędy (BEZ Quantity>0 i Price>0)
    mask = (
        df["InvoiceDate"].notna() &
        (df["InvoiceDate"] <= pd.Timestamp.now()) &
        df["Invoice"].ne("") &
        df["StockCode"].ne("")
    )
    df = df.loc[mask].copy()

    # Normalizacja opisów
    df["Description"] = df["Description"].replace({"nan": "", "None": ""}).fillna("").str.strip()

    # Total Value (może być ujemne!)
    df["TotalValue"] = df["Quantity"] * df["Price"]

    return df

clean_v2 = clean_raw_v2(raw)
clean_v1 = clean_raw(raw)

print(f"CLEAN_v1 (stara): {len(clean_v1):,}")
print(f"CLEAN_v2 (nowa): {len(clean_v2):,}")
print(f"Różnica: +{len(clean_v2) - len(clean_v1):,} rekordów")
    

[2025-09-01 15:22:28] Walidacja obecnosci kolumn RAW...
[2025-09-01 15:22:28] Konwersja typów (Quantity, Price, InvoiceDate)...
[2025-09-01 15:22:30] Deduplikacja rekordów...
[2025-09-01 15:22:30] Usunięto duplikatów: 34,337
[2025-09-01 15:22:30] Filtrowanie prawidłowych sprzedaży...
[2025-09-01 15:22:31] Po filtrowaniu: 1,007,912 rekordów
[2025-09-01 15:22:31] Standaryzacja krajów...
[2025-09-01 15:22:31] Czyszczenie opisów produktu...
[2025-09-01 15:22:31] Kalkulacja Total_Value...
[2025-09-01 15:22:31] Łączny przychód: £20,476,082.17
[2025-09-01 15:22:31] Quality Gates po czyszczeniu...
CLEAN_v1 (stara): 1,007,912
CLEAN_v2 (nowa): 1,032,767
Różnica: +24,855 rekordów
